# Week 3a.2 — The Agent Loop as a Graph

In Week 2b the loop was: model proposes a tool call, our code executes it, the result goes back, repeat until the model answers. `create_agent` ran it for us. Now we have the pieces to build that exact loop as a graph: `MessagesState`, nodes, and one conditional edge.

In [ ]:
from dotenv import load_dotenv
import os

load_dotenv()
assert os.getenv("GOOGLE_API_KEY"), "No GOOGLE_API_KEY found."
print("API key loaded")

## 0. Model Setup

Execute one of the following to define the model.

If using Ollama, you will need to start it first (simply open the chat UI and send a message):
- https://docs.langchain.com/oss/python/integrations/chat/ollama

Verify that Ollama is serving a model:
- http://localhost:11434/

In [ ]:
from langchain_ollama import ChatOllama

model = ChatOllama(model="qwen3.5:4b", reasoning=False)

In [ ]:
from langchain.chat_models import init_chat_model

model = init_chat_model(model="gpt-4.1-mini")

In [ ]:
from langchain_google_genai import ChatGoogleGenerativeAI

model = ChatGoogleGenerativeAI(model="gemini-3.5-flash-lite")

## 1. The tools from Week 2

Same three tools as the Week 2b agent: a clock, date arithmetic, and web search.

In [ ]:
from datetime import datetime
from langchain.tools import tool

@tool
def get_current_time() -> str:
    """Return the current local date and time."""
    return datetime.now().strftime("%A, %B %d, %Y at %I:%M %p")

@tool
def days_until(date: str) -> str:
    """Return the number of days from today until a future date given as YYYY-MM-DD."""
    target = datetime.strptime(date, "%Y-%m-%d").date()
    delta = (target - datetime.now().date()).days
    return f"{delta} days"

In [ ]:
from langchain_tavily import TavilySearch
from typing import Dict, Any

@tool
def search_the_web(query: str) -> Dict[str, Any]:
    """Search the web for information"""
    tavily = TavilySearch(max_results=3)
    return tavily.invoke({"query": query})

In [ ]:
tools = [get_current_time, days_until, search_the_web]

model_with_tools = model.bind_tools(tools)

## 2. The assistant node

The assistant node does one thing: hand the conversation to the model and append whatever comes back — an answer or a tool call. The system prompt is prepended on every call but never stored in the state.

In [ ]:
from langgraph.graph import MessagesState
from langchain.messages import SystemMessage, HumanMessage

sys_msg = SystemMessage(content="You're a helpful assistant who answers users' questions concisely. Use the tools available when necessary.")

def assistant(state: MessagesState) -> dict:
    return {"messages": [model_with_tools.invoke([sys_msg] + state["messages"])]}

## 3. The tool node

In Week 2b we executed tool calls by hand: read `tool_calls` from the last message, invoke the right tool, wrap the result in a `ToolMessage`. LangGraph ships that exact logic as a prebuilt node, `ToolNode`.

- https://reference.langchain.com/python/langgraph/agents/#langgraph.prebuilt.tool_node.ToolNode

In [ ]:
from langgraph.prebuilt import ToolNode

tool_node = ToolNode(tools)

## 4. Wire the loop

One decision drives the whole agent: after the assistant speaks, did it call a tool?

- If yes, go to the tool node.
- If no, the answer is final; end.

That condition is also prebuilt: `tools_condition` inspects the last message and routes to `"tools"` or to `END`.

Here is the complete loop we are wiring. Solid arrows are fixed edges; dashed red arrows are conditional:

![](figures/graph_agent_loop.png)

In [ ]:
from langgraph.graph import StateGraph, START, END
from langgraph.prebuilt import tools_condition

builder = StateGraph(MessagesState)

builder.add_node("assistant", assistant)
builder.add_node("tools", tool_node)

builder.add_edge(START, "assistant")
builder.add_conditional_edges("assistant", tools_condition)

#TODO: after the tools run, where should control go? Add that edge.


In [ ]:
agent = builder.compile()

from IPython.display import Image, display
display(Image(agent.get_graph().draw_mermaid_png()))

That edge back from `tools` to `assistant` is what turns a router into an agent: the model sees every tool result and decides again — another tool call or a final answer.

## 5. Run it

In [ ]:
result = agent.invoke({"messages": [HumanMessage(content="How many days until the midterm on 2026-10-30?")]})

for m in result["messages"]:
    m.pretty_print()

In [ ]:
result = agent.invoke({"messages": [HumanMessage(content="Who won the Super Bowl in 2026?")]})

print(result["messages"][-1].text)

In [ ]:
# This one needs two different tools; watch the loop run twice.
result = agent.invoke({"messages": [HumanMessage(content="What is today's date, and how many days until Thanksgiving on 2026-11-26?")]})

for m in result["messages"]:
    m.pretty_print()

## 6. `create_agent`, revisited

Compare with Week 2b's one-liner. Under the hood, `create_agent` builds essentially the graph you just wired: an assistant node, a tool node, `tools_condition`, and the loop-back edge. Use the prebuilt when the standard loop is all you need; build the graph yourself when it is not — which is exactly what Week 3b will require.

In [ ]:
from langchain.agents import create_agent

prebuilt_agent = create_agent(model=model, tools=tools, system_prompt=sys_msg.content)

result = prebuilt_agent.invoke({"messages": HumanMessage(content="How many days until the midterm on 2026-10-30?")})
print(result["messages"][-1].text)

## 7. ICA: your weather tool joins the graph

In Week 2b you wrote `get_weather(city)` against wttr.in. Add it to this agent:

1. Paste (or rewrite) your `get_weather` tool below.
2. Rebuild the tool list, rebind it to the model, and rebuild the graph.
3. Ask: "Should I bring a jacket to campus tonight?" and check which tools the agent used.

In [ ]:
import requests

@tool
def get_weather(city: str) -> str:
    """TODO: describe what this tool does and what the argument means."""
    # TODO: fetch https://wttr.in/<city>?format=j1 with requests.get
    # TODO: return the current_condition segment of the JSON
    pass


In [ ]:
#TODO: rebuild tools, model_with_tools, and the graph, then ask the question.
